# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VinayBhavikatti/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Content Lifecycle

The research paper reports that growing pages were younger on average than declining pages. It compares groups using measures such as content age, impressions, average position, and word count.

**Methodology question:**
How exactly is the growth/decline label constructed, and does the validation design separate information available before the measurement window from information observed during or after that window?

This matters because using future-period performance to define the label and then using overlapping performance signals as features could make the relationship look stronger than it would be in a real prediction setting.

### Finding 2 — Freshness Multiplier

The paper reports different growth-to-decline ratios across freshness windows and a separate comparison between refreshed and stale mature pages.

**Methodology question:**
Does the refreshed-versus-stale comparison control for the fact that pages chosen for refresh may already differ from stale pages in age, visibility, traffic, or other characteristics?

This is important because the observed difference can support a directional association, but it does not by itself prove that refreshing a page caused the improvement.

### Review framing

These questions are methodological checks rather than criticisms of the findings. They help distinguish an observed association from a causal claim and make the validation design easier to interpret.


## 2. My model under an honest split

The Week-5 model used a standard train/test setup and achieved very high precision. Because the target is related to content-performance signals, I am checking whether the result remains strong when pages from the same client are not allowed to appear in both training and testing.

I use a client-grouped split because content from the same client can share patterns that would make a random split overly optimistic.

The grouped result is treated as the more conservative estimate of generalization to unseen clients.

In [18]:
# Section 2 — Prepare modelling data

# Create the Week-5 target from the baseline decision rule.
# A page is treated as an action candidate when it has meaningful visibility
# and shows evidence of needing review.

# ============================================================
# SECTION 2 — PREPARE MODELLING DATA
# ============================================================

df = df.copy()

# ------------------------------------------------------------
# 1. Week-5 features
# ------------------------------------------------------------

week5_features = [
    "content_age_days",
    "impressions_90d",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

# ------------------------------------------------------------
# 2. Features excluded during leakage review
# ------------------------------------------------------------

leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

# Keep only Week-5 features that are available
# and are not considered leakage candidates.
safe_features = [
    c for c in week5_features
    if c in df.columns and c not in leakage_candidates
]

# ------------------------------------------------------------
# 3. Create Week-5 baseline target
# ------------------------------------------------------------

df["baseline_action"] = (
    (
        (df["days_since_last_update"] >= 180) &
        (df["impressions_90d"] >= 500)
    )
    |
    (
        (df["trend_direction"].astype(str).str.lower() == "down") &
        (df["impressions_90d"] >= 100)
    )
    |
    (
        (df["word_count"] > 0) &
        (df["word_count"] < 1200) &
        (df["impressions_90d"] >= 250)
    )
).astype(int)

# ------------------------------------------------------------
# 4. Prepare modelling dataframe
# ------------------------------------------------------------

required_columns = safe_features + [
    "baseline_action",
    "client_id"
]

model_df = df[required_columns].copy()

# Convert modelling features to numeric
for col in safe_features:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

# Remove rows with missing modelling values
model_df = model_df.dropna(
    subset=required_columns
)

# ------------------------------------------------------------
# 5. Define X, y and client groups
# ------------------------------------------------------------

X = model_df[safe_features]
y = model_df["baseline_action"].astype(int)
groups = model_df["client_id"]

# ------------------------------------------------------------
# 6. Validation checks
# ------------------------------------------------------------

print("=" * 60)
print("SECTION 2 — MODELLING DATA CHECK")
print("=" * 60)

print("\nWeek-5 features:")
print(week5_features)

print("\nFeatures excluded for leakage review:")
print(leakage_candidates)

print("\nSafe features used by the model:")
print(safe_features)

print("\nBaseline action distribution:")
print(df["baseline_action"].value_counts())

print("\nRows used for modelling:")
print(len(model_df))

print("\nTarget distribution after cleaning:")
print(y.value_counts())

print("\nNumber of clients:")
print(groups.nunique())

print("\nMissing values in modelling data:")
print(model_df[required_columns].isna().sum().sum())

print("\nSection 2 preparation complete.")

SECTION 2 — MODELLING DATA CHECK

Week-5 features:
['content_age_days', 'impressions_90d', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Features excluded for leakage review:
['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Safe features used by the model:
['content_age_days', 'impressions_90d', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Baseline action distribution:
baseline_action
0    16809
1    13191
Name: count, dtype: int64

Rows used for modelling:
19897

Target distribution after cleaning:
baseline_action
0    10101
1     9796
Name: count, dtype: int64

Number of clients:
29

Missing values in modelling data:
0

Section 2 preparation complete.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

A feature is considered suspicious when it contains information that would not realistically be available at prediction time or when it directly overlaps with the rule used to create the target.

My Week-5 target is `baseline_action`. Several dataset columns describe recent performance and trend behavior. Because the target is related to an action rule, these variables need to be checked rather than automatically treated as independent predictors.

I therefore separate ordinary candidate features from variables that may encode the target or its future outcome window.

In [19]:
# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

print("=" * 60)
print("SECTION 3 — LEAKAGE AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check which suspicious columns actually exist
# ------------------------------------------------------------

available_suspicious = [
    c for c in leakage_candidates
    if c in df.columns
]

missing_suspicious = [
    c for c in leakage_candidates
    if c not in df.columns
]

print("\nSuspicious columns found:")
print(available_suspicious)

print("\nSuspicious columns not found:")
print(missing_suspicious)


# ------------------------------------------------------------
# 2. Inspect suspicious columns
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("COLUMN-LEVEL AUDIT")
print("-" * 60)

for col in available_suspicious:

    print(f"\n--- {col} ---")
    print("dtype:", df[col].dtype)
    print("unique values:", df[col].nunique())
    print("missing values:", df[col].isna().sum())

    # Numeric association with baseline target
    if pd.api.types.is_numeric_dtype(df[col]):

        temp = df[[col, "baseline_action"]].dropna()

        if len(temp) > 0:
            correlation = temp[col].corr(
                temp["baseline_action"]
            )

            print(
                "Correlation with baseline_action:",
                round(correlation, 3)
            )

    else:
        # For categorical variables, show distribution
        print("Value counts:")
        print(df[col].value_counts(dropna=False).head(10))


# ------------------------------------------------------------
# 3. Check direct overlap with the target rule
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("TARGET-RULE OVERLAP CHECK")
print("-" * 60)

rule_components = [
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "word_count"
]

print("\nVariables used directly to construct baseline_action:")
print(rule_components)

print(
    "\nThese variables are not automatically leakage, "
    "but they are closely related to the target definition."
)


# ------------------------------------------------------------
# 4. Confirm conservative model features
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("CONSERVATIVE MODEL FEATURE CHECK")
print("-" * 60)

print("\nFeatures used by the model:")
print(safe_features)

print("\nExcluded leakage-review features:")
print(leakage_candidates)

overlap = set(safe_features).intersection(
    set(leakage_candidates)
)

print("\nLeakage-candidate features still used:")
print(list(overlap))

if len(overlap) == 0:
    print(
        "\nPASS: No leakage-review candidate is included "
        "in the conservative feature set."
    )
else:
    print(
        "\nWARNING: Some leakage-review candidates are still "
        "included in the model."
    )


# ------------------------------------------------------------
# 5. Final audit summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LEAKAGE AUDIT SUMMARY")
print("=" * 60)

print(
    "The conservative model excludes trend and recent-period "
    "outcome variables that could overlap with the target definition."
)

print(
    "The remaining features are treated as available input signals, "
    "but this audit does not prove that they are free from every "
    "possible form of temporal leakage."
)

print("\nSection 3 leakage audit complete.")

SECTION 3 — LEAKAGE AUDIT

Suspicious columns found:
['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Suspicious columns not found:
[]

------------------------------------------------------------
COLUMN-LEVEL AUDIT
------------------------------------------------------------

--- trend_direction ---
dtype: object
unique values: 5
missing values: 0
Value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

--- trend_pct ---
dtype: float64
unique values: 2712
missing values: 3388
Correlation with baseline_action: -0.102

--- impressions_last_30d ---
dtype: int64
unique values: 5182
missing values: 0
Correlation with baseline_action: -0.042

--- clicks_last_30d ---
dtype: int64
unique values: 239
missing values: 0
Correlation with baseline_action: -0.029

--- sessions_last_30d ---
dtype: int64
unique

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite and failure analysis

The Week-5 model produced false positives and false negatives even though the original precision was very high.

A false positive means the model predicted that a page belonged to the action class when the observed label was 0.

A false negative means the model predicted 0 when the observed label was 1.

These errors show that the available features do not perfectly determine the action label. In particular, pages with stable or mixed trend behavior can be difficult to classify correctly.

The errors reinforce that the model should be treated as decision-support rather than as an automatic decision maker.

In [22]:
import numpy as np
# ============================================================
# SECTION 4 — REAL FAILURE EXAMPLES
# ============================================================

print("=" * 55)
print("SECTION 4 — REAL FAILURE EXAMPLES")
print("=" * 55)

# Get the original row indices belonging to the grouped test set
test_original_idx = model_df.iloc[test_idx].index

# Build results from the original dataframe
group_results = df.loc[test_original_idx].copy()

# Add actual and predicted values
group_results["actual"] = y_test.values
group_results["predicted"] = grouped_pred

# Classify prediction errors
group_results["error_type"] = np.select(
    [
        (group_results["actual"] == 1) &
        (group_results["predicted"] == 0),

        (group_results["actual"] == 0) &
        (group_results["predicted"] == 1)
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

# Error summary
print("\nError summary:")
print(group_results["error_type"].value_counts())

# ------------------------------------------------------------
# False-negative examples
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FALSE-NEGATIVE EXAMPLES")
print("=" * 55)

false_negatives = group_results[
    group_results["error_type"] == "False Negative"
]

# Show only columns that actually exist
display_columns = [
    "content_id",
    "client_id",
    "actual",
    "predicted",
    "content_age_days",
    "impressions_90d",
    "search_volume",
    "ctr",
    "avg_position"
]

display_columns = [
    col for col in display_columns
    if col in false_negatives.columns
]

if len(false_negatives) > 0:
    display(false_negatives[display_columns].head(10))
else:
    print("No false-negative examples found.")

# ------------------------------------------------------------
# False-positive examples
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FALSE-POSITIVE EXAMPLES")
print("=" * 55)

false_positives = group_results[
    group_results["error_type"] == "False Positive"
]

display_columns = [
    "content_id",
    "client_id",
    "actual",
    "predicted",
    "content_age_days",
    "impressions_90d",
    "search_volume",
    "ctr",
    "avg_position"
]

display_columns = [
    col for col in display_columns
    if col in false_positives.columns
]

if len(false_positives) > 0:
    display(false_positives[display_columns].head(10))
else:
    print("No false-positive examples found.")

print("\nSection 4 complete.")

SECTION 4 — REAL FAILURE EXAMPLES

Error summary:
error_type
Correct           4169
False Negative     753
False Positive     741
Name: count, dtype: int64

FALSE-NEGATIVE EXAMPLES


,content_id,client_id,actual,predicted,content_age_days,impressions_90d,search_volume,ctr,avg_position
0,content_304f48230142,client_f369cb89fc,1,0,187,3803,10.0,0.76,10.6
1,content_a1fb4e703a9e,client_4e07408562,1,0,445,15320,90.0,0.05,20.3
127,content_02141810795a,client_4e07408562,1,0,487,1912,10.0,0.05,16.3
129,content_b4170c25efd2,client_4e07408562,1,0,421,2159,30.0,0.05,21.3
165,content_eaea09d6891e,client_4e07408562,1,0,545,1035,10.0,0.00,23.6
294,content_899bf6f73251,client_4e07408562,1,0,545,1536,110.0,0.07,11.8
386,content_e474339b073f,client_4e07408562,1,0,420,11106,20.0,0.24,24.2
412,content_61f4a24677e0,client_4e07408562,1,0,390,2441,170.0,0.16,12.7
413,content_74aa486106a1,client_4e07408562,1,0,390,14173,320.0,0.40,8.7
449,content_59ee92135b5c,client_7f2253d7e2,1,0,224,24978,0.0,1.15,13.9



FALSE-POSITIVE EXAMPLES


,content_id,client_id,actual,predicted,content_age_days,impressions_90d,search_volume,ctr,avg_position
26,content_72c5c2d73e5a,client_4e07408562,0,1,300,2426,0.0,0.12,30.0
34,content_55f75c034970,client_d029fa3a95,0,1,140,3998,0.0,0.03,6.4
64,content_685de0e3b7cb,client_f369cb89fc,0,1,106,2639,10.0,0.11,7.2
134,content_1ec660df5cc3,client_d029fa3a95,0,1,144,717,0.0,0.14,4.2
135,content_670746e86425,client_4e07408562,0,1,280,19802,20.0,0.32,20.4
181,content_722d8cd002d3,client_f369cb89fc,0,1,95,1220,0.0,0.41,13.9
189,content_fd270fc98fa0,client_d029fa3a95,0,1,148,111,0.0,0.90,5.2
204,content_976d5deeab73,client_4e07408562,0,1,445,5996,1000.0,0.10,19.0
236,content_47d2fed41749,client_d029fa3a95,0,1,203,519,0.0,0.00,7.4
296,content_7f418e12d343,client_d029fa3a95,0,1,130,190,0.0,0.00,6.6



Section 4 complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.